# Frequency Shift Keying Project - Part 1: Converting Text to Bits and Back

In [ ]:
## load numpy and matplotlib
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# get access to some unicode data
import unicodedata

## Background

A communications system aims to send information from one locations to
another. In a digital communications system, the information consists
of a sequence of bits. Simply put, a digital communications system
uses the following steps to accomplish that goal:

* The transmitter is provided a sequence of bits that must be sent
  to the receiver; the receiver is in a different location from the
  transmitter.
* The transmitter maps the sequence of bits into an analog signal
  or waveform.
* The signal is sent to the receiver; this may happen via a cabled
  connection or wirelessly. As the signal travels from transmitter to
  the receiver it becomes weaker and it is disturbed by noise.
* The receiver observes this signal and employs knowledge of how
  the transmitter maps bits to signals to reconstruct the original bit
  sequence. 

This lab focuses on the first step by exploring how textual
information can be converted to and from a sequence of bits.

## Information in Binary Format: Bits

Modern communication systems move information in binary format - as a
sequence of bits. Bits are binary because each bit can take on only
two possible values, denoted as $0$ and $1$.

Humans
don't "consume" information in binary format directly; instead we
rely on more complex representations of information; e.g., written
text.
This first part of the project explores how to convert textual information into a
sequence of bits and vice versa.

Consider the string `'Information'`; it is in a form that can
be understood by humans. However, it is not immediately suited for a digital
communications system that requires a sequence of bits. Let's explore
how to convert a text string into a sequence of bits in Python.

### ASCII

A long time ago, representing text used to be easy as the American Standard Code of Information Interchange (ASCII) allowed 
only a samll set of characters in strings. For exmple, the string `'Information'` consists entirely of ASCII characters.

There are 128 ASCII characters, including the characters you see on your keyboard. Each of these chracters is represented by a 7-bit pattern which can be expressed either as a decimal number between 0 and 127 or a hexadecimal number between `0x00` and `0x7F`. Recall that hexadecimal numbers use 16 digits and use base 16. 

The code below produces a table of (printable) ASCII characters and their codes in decimal and hexadecimal. It relies on the Python built-in function `chr` that turns an integer into the corresponding character. The inverse function for `chr` is `ord`.

In [ ]:
## make a simple ASCI table
print("Character     Code (decimal)   Code (hex)   Description")
for n in range(128):
    c = chr(n)
    try:
        print(f'    {c}          {n:3d}               {n:02x}         {unicodedata.name(c)}')
    except ValueError:
        pass

### Unicode

Today, textual information may be represented in non-English characters and may include emojis or other glyphs. Over the years,
the Uncode Transformation Format - 8-Bit (short UTF-8) has become prevalent. Using UTF-8, we can, for example, construct the string
`'Inf😲rmätion'`.

Fortunately, Python has excellent support for unicode or UTF-8.
Here are a few (interesting) Unicode characters.

In [ ]:
## make a table with some non-ASCII characters
print("Character     Code (decimal)   Code (hex)   Description")
for n in [233, 0x0bf2, 16234, 6000, 13231, 128562, 228]:
    c = chr(n)
    try:
        print(f'    {c}    \t  {n:06d}         {n:06x}     {unicodedata.name(c)}')
    except ValueError:
        pass

### From text to bits

Recall that our aim is to transform text into a sequence of bits. We will achieve that in two steps:
1. convert a (unicode) string into a sequence of bytes; this is accomplished with the method `encode()` that is defined for strings
2. convert the sequence of bytes into a sequence of bits

#### Encoding strings to bytes

Let's begin with the first step: converting a string into a sequence of bytes. For a given string `s`, we can obtain the sequence of bytes `b` that represent this string using `s.encode()`.

The inverse operation - coverting a sequence of bytes to a string - us called decoding and is performed using `b.decode("utf-8", "replace")`. The first argument indicates that that we want a Unicode string. The second parameter is relevant when some of the bits in the byte sequence have been corrupted. By specifying `"replace"` such errors do not lead to decoding errors, instead place-holder symbols � are inserted.

In [ ]:
# define a string variable (with some unicode characters)
s = 'Inf😲rmätion'

# turn the string into a sequence of bytes
b = s.encode()

# let's see what happened:
print(f'The string {s} is encoded as {b}')
print(f'The byte sequence can be decoded using `b.decode("utf-8", "replace")`: {b.decode("utf-8", "replace")}')

A byte sequence in Python is delimited by `b''`. Inside the quotation marks, bytes that can be printed as ASCII characters appear like regular characters (like the `Inf` at the beginning). 

Non-ASCII characters, like the smiley or the Umlaut, are represented by "raw" sequences of bytes; each byte consists of `\x` followed by two hexadecimal digits (`0` through `9` and `a` through `f`). 

Note that the smiley requires four bytes and the Umlaut takes two bytes. In contrast the "regular" characters require only one byte.

We can easily turn the byte-string `b` into an array of (decimal) integers between 0 and 255. We have two options:
1. Use list comprehension over the elements of the byte sequence. This takes advantage of the fact that a byte corresponds to a number between 0 and 255.
2. Using the NumPy function `np.frombuffer`. Here, we must specify that elements of the resulting array are of type `dtype=np.uint8` (unsigned 8-bit integer)

Let's try both of these methods:

In [ ]:
# using a list comprehension
bytes_list = [x for x in b]

# using np.frombuffer()
bytes_array = np.frombuffer(b, dtype=np.uint8)

print('bytes_list:', bytes_list)
print('bytes_array:', bytes_array)

Both methods yield the same answer. Note, that `bytes_list` is a Python `list` while `bytes_array` is a NumPy array.

Integers less than 128 correspond directly to ASCII characters. For example, `73` corresponds to letter `I` (see the ASCII table above).

The larger integers correspond to the hexadecimal representations. For example, the first byte of the smiley is `\xf0`. This is hexadecimal for `240`. Recall that in hexadecimal the symbol `f` has place value 15 and the base is 16. So that `f0` equals `15 * 16 + 0 = 240`.

#### Converting bytes to bits

Let's tackle the second step in our quest to reduce strings to a sequence of bits.

We must convert each of the bytes that the `encode()` function produced into a sequence of 8 bits. The function below does that using the bitwise **and** operator `&` and the bitwise-shift operators `>>` and `<<`.  

In [ ]:
def int_to_bits(b, R=8):
    """convert a positive integer into a sequence of bits
    
    Parameters:
    -----------
    b - integer, must be between 0 and 2^R-1
    R - number of bits (default: 8)

    Returns:
    --------
    NumPy array of length R with only 0s and 1s

    Raises:
    -------
    `ValueError` if n is not between 0 and 2^R-1

    The bits are stored in LSB first order; the least-significant bit is stored in the 
    first position of the resulting array
    """

    if b < 0 or b >= 2**R:
        raise(ValueError(f'{b} cannot be represented with {R} bits.'))
    
    # allocate storage for result; use the smallest possible element size
    res = np.zeros(R, dtype=np.uint8)

    # p = 1; sets the LSB to 1 and all other bits to 0
    p = 1
    for n in range(R):
        # Extract bit in the n-th poition; then, move it to the LSB
        res[n] = (b & p) >> n
        # shift the 1 in p one position to the left
        p = p << 1


    return res

In [ ]:
## Verify that this works:
# 122 = 2 + 8 + 16 + 32 + 64; result should be [0 1 0 1 1 1 1 0 0]
n = 122
print( f'bit vector for {n} is ', int_to_bits(n))

Study this function to understand how the bitwise and
(`&`) function is used to extract one bit at a time from
the integer. 

Also, understand, how the bit-shift operators
(`>>` and `<<`) are used to move the single 1-bit in `p` into the position
that is extracted by the and-operation.

#### Task: Convert bits to integers

Write a function `bits_to_int` that performs the inverse operation to
`int_to_bits`. 

In your function you are encouraged to use the bitwise **or** operator `|` and the bit-shift operators.

Your function must start like this:

In [ ]:
def bits_to_int(bit_seq):
    """Convert a list or array of bits to an integer
    
    Parameters:
    -----------
    bit_seq - a list or array of 0s and 1s

    Returns:
    --------
    positive integer

    Example:
    --------
    bits_to_int([1 0 1 1]) returns 13 (as 13 = 1 + 4 + 8)

    It is assumed that the first bit in `bit_seq` is the LSB
    """
    res = 0
    for n in range(len(bit_seq)):
        res += bit_seq[n] * 2**n

    return res
    

Use the cell below to verify that your function is correct.

In [ ]:
## unit tests
assert bits_to_int([1, 0, 1, 1]) == 13

## round-trip test: verify that the bits_to_int is the inverse of int_to_bits
for n in range(len(b)):
    assert bits_to_int( int_to_bits(b[n]) ) == b[n]

print('OK')

### Converting a string to a sequence of bits

All the pieces are now in place to convert a string into a sequence of
bits. Two steps are required:

1. First, each letter in the string is
converted into a positive integer. 
1. Then each integer is turned into
a sequence of bits. The bits for each letter are then concatenated to
produce the complete bit sequence that represents the string.

Below is a function that turns a given Unicode string into a sequence of
bits. 

In [ ]:
def string_to_bits( string):
    """Convert a unicode string to an array of bits
    
    Parameters:
    -----------
    string - the unicode string to be converted
    
    Returns:
    --------
    NumPy array of bits

    """

    # turn string into a sequence of bytes
    b = string.encode()

    # allocate storage for output: we need 8 bits for each byte
    # and store bits as unsigned 8-bit integers
    res = np.zeros(8*len(b), dtype=np.uint8)

    # convert each byte to bits and place in output array
    start = 0

    # loop over all bytes
    for c in b:
        # use function  int_to_bits to get bits
        res[start : start+8] = int_to_bits(c)
        start = start + 8

    
    return res

In [ ]:
## try it out
bits = string_to_bits(s)

print(f'The string `{s}` is represented by bits \n{bits}')

### Task: Convert  a bit sequence into a string

Write a function `bits_to_string` that performs the inverse operation to
`string_to_bits`. 

The function should also raise a ValueError as indicated in the docstring.

**Hint:** Use your `bits_to_int` function to process one byte at a time into an integer. Store these integers in a numpy array with `dtype=np.uint8`.

A numpy array of integers `int_array` can be converted to raw bytes using `bytes = int_array.tobytes()`. 

Raw bytes `bytes` can be decoded to a UTF-8 string using `bytes.decode("utf-8", "replace")`.

In [ ]:
def bits_to_string(bits):
    """Convert a sequence of bits to a string
    
    Parameters:
    -----------
    bits - NumPy array of bits

    Returns:
    --------
    decoded string

    Raises:
    -------
    ValueError if length of bits is not a multiple of 8.
    """

    if ((len(bits) % 8) != 0):
        raise ValueError('not a multiple of 8')

    res = np.zeros(len(bits)//8, dtype = np.uint8)

    for n in range(len(res)):
        res[n] = bits_to_int(bits[8*n : 8*n + 8])


    raw_bytes = res.tobytes()

    return raw_bytes.decode("utf-8", "replace")

    


In [ ]:
## Roundtrip check
assert bits_to_string( string_to_bits(s) ) == s

print("Decoding of bits produces: ", bits_to_string(bits))

## Questions

In addition to completing the programming exercises above, also answer the following questions for your report: 

### Bit Errors

What happens if some of the bits are corrupted? The string that we have been testing with produces 120 bits. What happens if one of these bits is "flipped"? Does it make a difference if the LSB or the MSB of a byte is flipped? Does it make a difference if one of the ASCII characters or the smiley character is affected by the bit error?

**Hint:** To "flip" the n-th bit of array `bits`, you can use `bits[n] = 1 - bits[n]`.

Use additional code and text cells to answer this question and demonstrate your points.

In [ ]:
## code to address this question
before_flip = string_to_bits(s)

def flip_bits(n):
    after_flip = string_to_bits(s)
    after_flip[n] = 1- after_flip[n]
    print(bits_to_string(after_flip))

print(s)

flip_bits(0)

flip_bits(7)

### Unicode encoding

The smiley emoji that we saw above is produced by `chr(0x1f632)`. However, the byte sequence that results from calling `decode` on this Unicode character is `b'\xf0\x9f\x98\xb2'`, i.e., the 4-byte sequence `0xf09f98b2`. Clearly, these two hexadecimal representations are very different.

Research the topic of Unicode encoding to figure out the relationship between the two hexadecimal numbers `0x1f632` and `0xf09f98b2`. Explain the difference between these two representations, and why there are two ways to represent it. Cite at least one source (a link to a reliable website is sufficient); AI is not a source.

In [ ]:
smiley = chr(0x1f632)
print("The smiley character is produced by `chr(0x1f632)`: ", smiley)
print("The UTF-8 encoding for the smiley character is: ", smiley.encode())

the unicode identifies the emoji but its UTF-8 encoding is 0xf09f98b2 which is a 4 byte sequence this is so a computer can store and transmit it unicodes tells us which character we are using and UTF-8 is how we store it in memory in bytes. Source(https://www.unicode.org/glossary/#UTF_8, https://www.unicode.org/faq/utf_bom.html)

## Summary

This was Part 1 of the Frequency Shift Keying (FSK) project. In this part of the project, we developed code to convert UTF-8 encoded strings into an array of bits and vice versa. No signal processing operations were performed in this Lab. Part 2 of the project will involve signal processing, as we will implement part of an FSK transmitter, which generates an FSK signal from an array of bits. We will also plot the spectrogram of an FSK signal.

### Deliverable

Submit a PDF version of this notebook. Make sure that:
* All markdown cells are properly typeset.
* All code cells are complete (no `FILL_ME_IN` left) and have been run in order.
* All functions are properly documented.
* All plots have proper axis labels and other adornments as appropriate.
* Answers are provided for the section entitled "Questions."